In [1]:
# %%
# STEP 2 — IMPORT LIBRARIES
# --------------------------------------------------
import pandas as pd
from pathlib import Path

print("Libraries loaded.")

Libraries loaded.


In [2]:
# %%
# STEP 3 — LOAD EPC DATASET AND CREATE LOOKUP
# --------------------------------------------------
epc_path = Path("/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_lat_long_4_dec.csv")

epc_df = pd.read_csv(epc_path)

# Ensure BUILDING_REFERENCE_NUMBER is string for matching folder names
epc_df["BUILDING_REFERENCE_NUMBER"] = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str)

# Create lookup dictionary: {building_id: PV_UPGRADE}
pv_lookup = dict(zip(epc_df["BUILDING_REFERENCE_NUMBER"], epc_df["PV_UPGRADE"]))

print(f"EPC dataset loaded. {len(pv_lookup)} buildings mapped.")

EPC dataset loaded. 117 buildings mapped.


In [3]:
# %%
# STEP 4 — DEFINE BASE DIRECTORY AND GET BUILDINGS
# --------------------------------------------------
base_dir = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

building_dirs = [d for d in base_dir.iterdir() if d.is_dir()]

print(f"Found {len(building_dirs)} building folders.")

Found 117 building folders.


In [4]:
# %%
# STEP 5 — PROCESS BUILDINGS WITH CONDITIONAL MAPPING
# --------------------------------------------------
success_count = 0
fail_count = 0
missing_epc_count = 0

for building_dir in building_dirs:
    
    building_id = building_dir.name
    print(f"\nProcessing building: {building_id}")
    
    # Check EPC lookup
    if building_id not in pv_lookup:
        print("No EPC entry found. Skipping.")
        missing_epc_count += 1
        continue
    
    pv_upgrade_flag = pv_lookup[building_id]
    
    # Conditional column mapping
    if pv_upgrade_flag == 0:
        column_mapping = {"pv_power": "existing_pv"}
    elif pv_upgrade_flag == 1:
        column_mapping = {"pv_power": "pv"}
    else:
        print(f"Unexpected PV_UPGRADE value: {pv_upgrade_flag}. Skipping.")
        fail_count += 1
        continue
    
    heating_file = building_dir / "TOTAL" / "pv_power.csv"
    energy_file = building_dir / "TOTAL" / "energy_results.csv"
    
    # Check files exist
    if not heating_file.exists() or not energy_file.exists():
        print("Missing required files. Skipping.")
        fail_count += 1
        continue
    
    try:
        # Load data
        heat_df = pd.read_csv(heating_file)
        energy_df = pd.read_csv(energy_file)
        
        # Validate columns
        missing_source = [col for col in column_mapping if col not in heat_df.columns]
        missing_target = [col for col in column_mapping.values() if col not in energy_df.columns]
        
        if missing_source:
            print(f"Missing source columns: {missing_source}")
            fail_count += 1
            continue
        
        # Initialize target columns if missing
        for target_col in column_mapping.values():
            if target_col not in energy_df.columns:
                energy_df[target_col] = pd.NA
        
        # Copy data safely
        for source_col, target_col in column_mapping.items():
            
            source_values = heat_df[source_col].values
            target_length = len(energy_df)
            source_length = len(source_values)
            
            copy_length = min(source_length, target_length)
            
            energy_df.loc[:copy_length-1, target_col] = source_values[:copy_length]
        
        # Save updated file
        energy_df.to_csv(energy_file, index=False)
        
        print(f"Success (PV_UPGRADE = {pv_upgrade_flag})")
        success_count += 1
    
    except Exception as e:
        print(f"Error: {e}")
        fail_count += 1

print("\n--- SUMMARY ---")
print(f"Successful: {success_count}")
print(f"Failed: {fail_count}")
print(f"Missing EPC entries: {missing_epc_count}")


Processing building: 1001664924
Success (PV_UPGRADE = 1)

Processing building: 1001829085
Success (PV_UPGRADE = 1)

Processing building: 1001063639
Success (PV_UPGRADE = 1)

Processing building: 1001829071
Success (PV_UPGRADE = 1)

Processing building: 1234761002
Success (PV_UPGRADE = 1)

Processing building: 1002539407
Success (PV_UPGRADE = 1)

Processing building: 1001063637
Success (PV_UPGRADE = 1)

Processing building: 1001664941
Success (PV_UPGRADE = 1)

Processing building: 1001991633
Success (PV_UPGRADE = 1)

Processing building: 1235057414
Success (PV_UPGRADE = 0)

Processing building: 1001829079
Success (PV_UPGRADE = 0)

Processing building: 1001664922
Success (PV_UPGRADE = 1)

Processing building: 1234761003
Success (PV_UPGRADE = 1)

Processing building: 1001664925
Success (PV_UPGRADE = 1)

Processing building: 1000238907
Success (PV_UPGRADE = 1)

Processing building: 1234761004
Success (PV_UPGRADE = 1)

Processing building: 1002313096
Success (PV_UPGRADE = 0)

Processing bu